In [2]:
#BGE sample

from FlagEmbedding import FlagModel

model = FlagModel(
    'BAAI/bge-large-zh-v1.5',
    query_instruction_for_retrieval="为这个句子生成表示以用于检索：",
    use_fp16=True
)

sentences_1 = ["我爱自然语言处理", "我喜欢机器学习"]
sentences_2 = ["我热爱BGE模型", "我关注文本检索"]

embeddings_1 = model.encode(sentences_1)
embeddings_2 = model.encode(sentences_2)

import numpy as np
similarity = np.matmul(embeddings_1, embeddings_2.T)
print(similarity)


/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[[0.4207 0.4873]
 [0.519  0.562 ]]


In [29]:
#统一处理数据
import os, json, random
from glob import glob

random.seed(42)

# === 输入输出路径 ===
DATA_DIR = "CSTS"
OUT_DIR = "CSTS_BGE"
os.makedirs(OUT_DIR, exist_ok=True)

def read_txt_file(path):
    """读取 CSTS 的 txt 文件，返回 [(s1, s2, label), ...]"""
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            s1, s2, label = parts[0].strip(), parts[1].strip(), parts[2].strip()
            if not s1 or not s2:
                continue
            # 部分文件可能label是'1.0'或'True'等，统一为0/1
            label = label.strip().lower()
            if label in ['1', '1.0', 'true', 't']:
                lab = 1
            elif label in ['0', '0.0', 'false', 'f']:
                lab = 0
            else:
                try:
                    lab = int(float(label) > 0.5)
                except:
                    continue
            pairs.append((s1, s2, lab))
    return pairs


def build_jsonl(split='train'):
    """整合 AFQMC, LCQMC, OPPO-xiaobu 三个子集生成统一 JSONL"""
    all_samples = []
    for sub in ['AFQMC', 'LCQMC', 'OPPO-xiaobu']:
        path = os.path.join(DATA_DIR, sub, f'{split}.txt')
        if not os.path.exists(path):
            print(f"⚠️ {path} 不存在，跳过")
            continue
        data = read_txt_file(path)
        all_samples.extend(data)
        print(f"✅ 读取 {sub}/{split}.txt: {len(data)} 条")

    print(f"➡️ 合并后 {split} 样本总数：{len(all_samples)}")

    # === 构建对比学习格式 ===
    output_path = os.path.join(OUT_DIR, f'{split}.jsonl')
    with open(output_path, 'w', encoding='utf-8') as f:
        for i, (s1, s2, lab) in enumerate(all_samples):
            f.write(json.dumps({
                        "sentence1": s1,
                        "sentence2": s2,
                        "label": lab
                    }, ensure_ascii=False) + '\n')


    print(f"✅ 已保存 {output_path}")
    print("—"*30)

# === 执行 ===
build_jsonl('train')
build_jsonl('dev')


✅ 读取 AFQMC/train.txt: 34334 条
✅ 读取 LCQMC/train.txt: 238766 条
✅ 读取 OPPO-xiaobu/train.txt: 167168 条
➡️ 合并后 train 样本总数：440268
✅ 已保存 CSTS_BGE/train.jsonl
——————————————————————————————
✅ 读取 AFQMC/dev.txt: 4316 条
✅ 读取 LCQMC/dev.txt: 8802 条
✅ 读取 OPPO-xiaobu/dev.txt: 10000 条
➡️ 合并后 dev 样本总数：23118
✅ 已保存 CSTS_BGE/dev.jsonl
——————————————————————————————


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import json

# ====== 1. 加载模型 ======

model_name = "BAAI/bge-base-zh-v1.5"
model = SentenceTransformer(model_name)

# ====== 2. 加载训练数据 ======
train_path = "CSTS_BGE/train.jsonl"
dev_path = "CSTS_BGE/dev.jsonl"

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            s1, s2, label = obj["sentence1"], obj["sentence2"], float(obj["label"])
            data.append(InputExample(texts=[s1, s2], label=label))
    return data

train_samples = load_jsonl(train_path)
dev_samples = load_jsonl(dev_path)

# ====== 3. 构造 DataLoader ======
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

# ====== 4. 构造验证器（可选） ======
dev_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    dev_samples, name="csts-dev"
)

# ====== 5. 开始微调 ======
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=dev_evaluator,
    epochs=3,                     # 训练轮数，可调整
    warmup_steps=100,
    output_path="bge-base-zh-v1.5-csts-finetuned",
    evaluation_steps=200,
    save_best_model=True,
    show_progress_bar=True

)


Step,Training Loss,Validation Loss,Csts-dev Pearson Cosine,Csts-dev Spearman Cosine
200,No log,No log,0.578349,0.596768
400,No log,No log,0.584579,0.604627
600,0.149000,No log,0.589077,0.609520
800,0.149000,No log,0.603150,0.617397


KeyboardInterrupt: 

In [36]:
import os
import json
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

# ========== 环境准备 ==========
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # 使用清华镜像加速 Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ========== 设备检测 ==========
use_gpu = torch.cuda.is_available()
device = "cuda" if use_gpu else "cpu"
print(f"🟢 检测到设备: {device}")

if use_gpu:
    print("GPU 信息：")
    os.system("nvidia-smi")

# ========== 模型选择 ==========
try:
    model_name = "BAAI/bge-large-zh-v1.5"
    model = SentenceTransformer(model_name, device=device)
    model.half()
    print("✅ 使用模型：BGE-Large (FP16)")
except RuntimeError as e:
    print("⚠️ 显存不足，自动切换为 BGE-Base")
    model_name = "BAAI/bge-base-zh-v1.5"
    model = SentenceTransformer(model_name, device=device)
    model.half()

# ========== 数据加载函数 ==========
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            # 兼容不同字段名
            s1 = obj.get("sentence1") or obj.get("text1") or obj.get("query")
            s2 = obj.get("sentence2") or obj.get("text2") or obj.get("reply")
            label = obj.get("label") or obj.get("score") or obj.get("gold_label")
            try:
                label = float(label)
            except:
                continue
            if s1 and s2:
                data.append(InputExample(texts=[s1, s2], label=label))
    print(f"✅ 加载 {len(data)} 条样本：{os.path.basename(path)}")
    return data

# ========== 数据路径 ==========
train_path = "CSTS_BGE/train.jsonl"
dev_path = "CSTS_BGE/dev.jsonl"

train_samples = load_jsonl(train_path)
dev_samples = load_jsonl(dev_path)

# ========== DataLoader 设置 ==========
# 动态 batch size 调整
batch_size = 16 if "large" not in model_name else 8
num_workers = min(4, os.cpu_count() or 1)

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=batch_size, num_workers=num_workers)
train_loss = losses.CosineSimilarityLoss(model)

# ========== 验证集评估器 ==========
dev_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(dev_samples, name="csts-dev")

# ========== 训练参数 ==========
epochs = 2
warmup_steps = 100
output_dir = f"{model_name.split('/')[-1]}-csts-finetuned"

print(f"\n🚀 开始训练：")
print(f"模型：{model_name}")
print(f"批次大小：{batch_size} | Epochs：{epochs} | Workers：{num_workers}\n")

# ========== 模型训练 ==========
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=dev_evaluator,
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path=output_dir,
    evaluation_steps=500,
    save_best_model=True,
    show_progress_bar=True
)

print(f"\n✅ 训练完成！模型已保存到：{output_dir}")

# ========== 测试示例 ==========
test_sentences = ["我喜欢机器学习", "我热爱人工智能"]
emb = model.encode(test_sentences)
print(f"样例相似度：{torch.cosine_similarity(torch.tensor(emb[0]), torch.tensor(emb[1]), dim=0).item():.4f}")


🟢 检测到设备: cuda
GPU 信息：
Tue Nov  4 09:39:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080 Ti     Off |   00000000:00:03.0 Off |                  N/A |
| 30%   27C    P8             12W /  350W |    4313MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------

3426.20s - Could not connect to 127.0.0.1: 33547
Traceback (most recent call last):
3426.23s - Could not connect to 127.0.0.1: 33547
  File "/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_comm.py", line 531, in start_client
    s.connect((host, port))
    ~~~~~~~~~^^^^^^^^^^^^^^
ConnectionRefusedError: [Errno 111] Connection refused
3426.28s - Could not connect to 127.0.0.1: 33547
Traceback (most recent call last):
  File "/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_comm.py", line 531, in start_client
    s.connect((host, port))
    ~~~~~~~~~^^^^^^^^^^^^^^
Traceback (most recent call last):
ConnectionRefusedError: [Errno 111] Connection refused
  File "/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_comm.py", line 531, in start_client
    s.connect((host, port))
    ~~

ConnectionRefusedError: [Errno 111] Connection refused

can only test a child process


ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

KeyboardInterrupt: 